# Aufnahme und Wiedergabe

## Overlay + Biblioteken + Funktionen
Das Overlay mit alle dazugehörigen Biblioteken und Funktionen und Treiber werden geladen. 

In [ ]:
# Overlay und Funktionen für Aufnahme und Filterung
from Filter_Overlay import*
# Um Audio im Notebook darstellen zu können
from IPython.display import Audio as IPAudio

In [ ]:
# Initialisierung des Audio-Codec
pAudio = init_codec(48000)

In [ ]:
# Ausgangslautstärke Wählen
pAudio.set_volume(30)
# Einstellen eingang: LineIn
pAudio.select_line_in()

In [ ]:
# Ausgangslautstärke Wählen
pAudio.set_volume(30)
# Einstellen eingang: HP/MIC
pAudio.select_microphone()

### Aufnahme:
Die Aufnahmezeit des Audiosignals kann im Bereich von 1 bis maximal 60 Sekunden frei gewählt werden, wobei diese Begrenzung durch den verwendeten Codec-Treiber vorgegeben ist. Aus Gründen der Systemstabilität wird jedoch empfohlen, die Aufnahmezeit auf maximal 30 Sekunden zu begrenzen.

In [ ]:
# Aufnahmezeit
recTime = 20
print("record Start")
pAudio.record(recTime)
print("record End")

# # in pAudio.buffer werden die aufnahmen gespreichert
print(pAudio.buffer, type(pAudio.buffer))

In [ ]:
# Buffer ausspielen über HP
pAudio.play()

In [ ]:
# speichert den Inhalt des Buffers als .wav
pAudio.save("record_file.wav")

In [ ]:
# Darstellen der .wav im Notebook
IPAudio("record_file.wav")

## Filter:
| Name:    | Filter:  | Typ:     | Fc:      | Ordnung: |
| -------- | -------- | -------- | -------- | -------- |
| *Filter_1* | **Hochpass** | Butterworth | 1kHz         | 2 |
| *Filter_2* | **Tiefpass** | Butterworth | 1kHz           | 2 |
| *Filter_3* | **Bandpass** | Butterworth | 500Hz - 2kHz   | 2 |
| *Filter_4* | **Bansstop** | Butterworth | 500Hz - 2kHz   | 2 |

Es wurden mehrere Filter implementiert, die auf Audiodateien angewendet werden können. Die Auswahl des gewünschten Filters erfolgt über die Funktion: *UseFilter(input-File, output-File, Filter-Name)*.<br>

Dabei wird als Eingabe die zu filternde Audiodatei angegeben, gefolgt vom gewünschten Namen der Zieldatei sowie dem Namen des anzuwendenden Filters. Nach Ausführung der Funktion wird das gefilterte Audiosignal automatisch unter dem angegebenen Ausgabedateinamen gespeichert.<br>

Die verfügbaren Filter sind durch ihre jeweiligen Bezeichnungen in der Tabelle auswählbar und können flexibel angewendet werden.<br>

Zusätzlich steht eine Funktion zur Mehrfachfilterung zur Verfügung, mit der sich ein Filter kaskadiert anwenden lässt. Dabei wird das Audiosignal mehrfach nacheinander durch denselben Filter geleitet, bevor es gespeichert wird.<br>
Die Anzahl der Durchläufe wird über einen einstellbaren Faktor bestimmt. Je höher dieser Faktor gewählt wird, desto stärker ist der Filtereffekt Es ist zu beachten, dass bei zu hohen Werten Instabilitäten oder Verzögerungen im Jupyter-Notebook auftreten können.<br>
*UseFilterCascade(input-File, output-File, Filter-Name, Faktor)* <br>

**Wichtig:** *Die Grenze für Audiodateien liegt bei etwa 20 MB. Alles darüber kann dazu führen, dass das Notebook abstürzt.*<br>

Das Board nimmt Audiosignale mit einer Auflösung von 24 Bit bei einer Abtastrate von 48 kHz auf. Entsprechend wurden auch alle implementierten Filter auf diese Samplingrate ausgelegt.<br>
Da viele digitale Audioquellen in der Regel mit 16 Bit und einer Abtastrate von 44,1 kHz vorliegen,  diese Formate automatisch anpasst. Werden Audiodateien mit 44,1 kHz Samplingrate als Eingang verwendet, erfolgt vor der Filterung ein automatisches Upsampling auf 48 kHz. Erst danach wird der Filter angewendet. Das Upsampling wird softwareseitig durchgeführt.<br>

In [ ]:
# Auswählen der zu filternen Datei:
#Audio_in = "record_file.wav"
Audio_in = "TF2_Main_Theme.wav"

# Ausangsnamen wählen:
Audio_out = "Filter_Out.wav"
Audio_out_cascade = "Filter_Out_cascade.wav"

In [ ]:
# Darstellen der .wav im Notebook
IPAudio(Audio_in)

In [ ]:
# Filter Anwenden:
#UseFilter(Audio_in, Audio_out,Filter_1) # HP
#UseFilter(Audio_in, Audio_out,Filter_2) # TP
#UseFilter(Audio_in, Audio_out,Filter_3) # BP
UseFilter(Audio_in, Audio_out,Filter_4) # BS

# Kaskadierte Filter Anwenden:
#UseFilterCascade(Audio_in,Audio_out_cascade,Filter_4,2)

## Ausgabe:
Für die Wiedergabe der verarbeiteten Audiodaten stehen im Notebook zwei Möglichkeiten zur Verfügung:
1. **Analoge Ausgabe über den Audioausgang des PYNQ-Boards:**<br> Bei dieser Variante wird die Audiodatei zunächst in den internen Buffer des Audio-Codecs geladen. Anschließend erfolgt die Ausgabe über den analogen Ausgang des Boards durch Aufruf einer separaten Abspielfunktion.

2. **Direkte Wiedergabe im Notebook über ein Audio-Widget:** <br> Alternativ kann direkt im Notebook über ein eingebettetes Audio-Widget wiedergegeben werden.

**Wichtig:** *Das Abspielen größerer Audiodateien über den analogen Ausgang kann zu Instabilitäten führen. Es wird daher empfohlen, solche Dateien nur direkt im Notebook wiederzugeben.*

In [ ]:
# Läd .wav in den Buffer
pAudio.load(Audio_out)

In [ ]:
# Buffer ausspielen über HP
pAudio.play()

In [ ]:
# Darstellen der .wav im Notebook nach der Filterung
IPAudio(Audio_out)
# IPAudio(Audio_out_cascade)

## Analyse:
Zur Analyse des Frequenzinhalts steht im Notebook eine Spektrumanalysefunktion zur Verfügung. Standardmäßig wird dabei das Spektrum der ersten 30 Sekunden des linken Audiokanals geplottet. Diese Parameter können jedoch bei Bedarf angepasst werden.<br>

*plot_spectrum(wav_path, time=30, channel=0)*<br>

**Prameter:**
- *time:* Gibt die zu analysierende Dauer in Sekunden an.
- *channel:* Bestimmt den Audiokanal (0 = linker Kanal, 1 = rechter Kanal).

**Wichtige Hinweise:**
Das Plotten von mehr als 30 Sekunden kann zu Instabilitäten oder Abstürzen des Jupyter-Notebooks führen. Daher wird empfohlen, pro Plot nicht mehr als 30 Sekunden anzuzeigen.<br>

Ist das eingelesene Datei kürzer als die angegebene Zeit, wird automatisch die maximale verfügbare Dauer verwendet.<br>

Die Darstellung erfolgt in normierter Form: Die Amplitude wird so skaliert, dass der größte Wert bei 0 dB liegt. Dadurch kann es vorkommen, dass Plots in ihrer absoluten Lautstärke unterschiedlich erscheinen, sie sind jedoch relativ zueinander vergleichbar.<br>

In [ ]:
# Darstellung des Eingangssignals:
plot_spectrum(Audio_in)

In [ ]:
# Darstellung des Ausgangssignals:
plot_spectrum(Audio_out)

In [ ]:
# Darstellung des kaskadierten Ausgangssignals:
plot_spectrum(Audio_out_cascade)